In [1]:
import threading
import time
from pathlib import Path

import torch
import numpy as np
import tqdm

from diff_mfld.geodesic.geodesic_funcs import ExpMethod, LogMethod
from diff_mfld.geometry.metric import MetricField
from diff_mfld.mfld import ComputeMfld
from optim.constr_methods import ConstrSolverMethod
from optim.constrained.ralm import RalmCfg
from optim.constrained.rsqo import RsqoCfg
from optim.subsolver_methods import SubsolverMethod
from optim.subsolvers.rgd import RiemGradDescentCfg
from optim.subsolvers.rtr import RiemTrustRegionCfg
from problem import create_problem

In [2]:
methods = [
    # ((ExpMethod.IVP, LogMethod.BVP), "ivpbvp"),
    ((ExpMethod.APPROX_O1, LogMethod.APPROX_O1), "o1"),
    ((ExpMethod.APPROX_O2, LogMethod.APPROX_O2), "o2"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O3), "o3"),
    ((ExpMethod.APPROX_O2, LogMethod.APPROX_O1), "o2_o1"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O1), "o3_o1"),
    ((ExpMethod.APPROX_O3, LogMethod.APPROX_O2), "o3_o2"),
]

# linear scaling of the domain
max_domain_scaling = [
     1., 2., 3., # 10. # 0.5
]

# parameters for random starting positions
p0_mean = np.array([2., -5., -8.])
p0_covar = 2. ** 2 * np.eye(3)
num_p0 = 10

# cost minimized at this point
cost_center = np.array([-5., -1., 3.])
cntr_center = np.array([-2., 0., 4.])
cntr_radius = 1.5


# the various metrics (describing the given curvature of the surface of optimization)

def euclid_metric(x1, x2, x3, scaling: float):
    metric = torch.eye(3)

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def scaled_metric(x1, x2, x3, scaling: float):
    # elements are assigned to this metric directly to preserve gradient history
    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.  # constant prevents degeneracy at origin
    metric[1, 1] = x2 ** 2 + 1.
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def coupled_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[0, 1] = 0.5 * x1 * x2
    metric[0, 2] = 0.5 * x1 * x3

    metric[1, 0] = 0.5 * x2 * x1
    metric[1, 1] = x2 ** 2 + 1.
    metric[1, 2] = 0.5 * x2 * x3

    metric[2, 0] = 0.5 * x3 * x1
    metric[2, 1] = 0.5 * x3 * x2
    metric[2, 2] = x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


def asymmetric_metric(x1, x2, x3, scaling: float):
    factor = 1. / scaling ** 2

    metric = torch.zeros((3, 3))
    metric[0, 0] = x1 ** 2 + 1.
    metric[1, 1] = 0.5 * x1 ** 2 * x2 ** 2 + 1.
    metric[2, 2] = 0.25 * x1 ** 2 * x2 ** 2 * x3 ** 2 + 1.

    # pulls the metric "back" onto the scaled coordinate system
    factor = 1. / scaling ** 2
    metric = factor * metric # avoids in-place operation to keep pytorch gradients

    return metric


metric_funcs = [
    (euclid_metric, "euclid"),
    (scaled_metric, "scaled"),
    (coupled_metric, "coupled"),
    (asymmetric_metric, "asymmetric"),
]


In [3]:
# generates the random points and scales them

# "To Everything? To the great Question of Life, the Universe and everything?"
r = np.random.default_rng(42)
p_rand_starting = r.multivariate_normal(p0_mean, p0_covar, num_p0)
max_starting_norm = np.max(np.linalg.vector_norm(p_rand_starting, axis=1))

# scales the optimization functions and starting points
p_rand_starting /= max_starting_norm
cost_center /= max_starting_norm
cntr_center /= max_starting_norm
cntr_radius /= max_starting_norm

In [4]:
# configure the constrained configurations
rsqo_cfg = RsqoCfg()

optimizers = [
    ((ConstrSolverMethod.RSQO, rsqo_cfg), "rsqo"),
]

In [5]:
# root directory to store output files inside
base_data_dir = Path("../data/constrained")

In [6]:
from diff_mfld.mfld import Mfld
import itertools

# run the optimization scheme

cfgs = itertools.product(
    metric_funcs, max_domain_scaling, optimizers, methods
)
cfg_pbar = tqdm.tqdm(cfgs, desc="Configurations")

for (
        (metric_fn, metric_fn_label),
        scaling,
        ((optimizer, optimizer_cfg), optimizer_label),
        ((exp_method, log_method), method_label)
) in cfg_pbar:

    # setup the manifold configuration
    metric = MetricField(lambda x1, x2, x3: metric_fn(x1, x2, x3, scaling=scaling))
    conn = metric.christoffels()  # Levi-Civita connection (metric-compatible and torsion-free)
    mfld = Mfld(metric, conn)

    compute_mfld = ComputeMfld(
        mfld=mfld,
        exp_method=exp_method,
        log_method=log_method,
        dist_method=log_method,  # which logarithm to use when computing distance
    )

    # scale the problem coordinates accordingly
    trials_p_starting = p_rand_starting * scaling
    trials_cost_center = cost_center * scaling

    trials_region_center = cntr_center * scaling
    trials_region_radius = cntr_radius * scaling

    # create the problem
    cost, region_ineqs = create_problem(torch.tensor(trials_cost_center), [[(torch.tensor(trials_region_center), trials_region_radius)]])
    region_ineq = region_ineqs[0]

    trials_pbar = tqdm.tqdm(range(trials_p_starting.shape[0]), desc="Trials")
    for start_idx in trials_pbar:
        p0 = torch.tensor(trials_p_starting[start_idx])
        result = optimizer(cost, [region_ineq], [], p0, compute_mfld, optimizer_cfg)
        print(result)

        # save the resulting dataclass
        filename = f"{optimizer_label}/metric_{metric_fn_label}__scaling_{scaling}__trial_{start_idx}__geod_method_{method_label}.pt"
        file_path = base_data_dir / filename
        print(f"\tSaving to {file_path}")

        torch.save(result, file_path)

Configurations: 0it [00:00, ?it/s]
Trials: 100%|██████████| 10/10 [00:00<00:00, 258.31it/s]


RsqoResult(success=True, p=tensor([-0.2643, -0.0414,  0.2369]), iters=2, history=RsqoHistory(p_hist=tensor([[-0.3398, -0.0752,  0.1996],
        [-0.2643, -0.0414,  0.2369]]), f_hist=tensor([8.8584e-05, 4.2724e-03]), gs_hist=tensor([[0.0412],
        [0.0082]]), hs_hist=tensor([], size=(2, 0)), g_mults_hist=tensor([[0.],
        [0.]]), h_mults_hist=tensor([], size=(2, 0)), penalty_hist=tensor([1.2000, 1.2000])))
	Saving to ../data/constrained/rsqo/metric_euclid__scaling_1.0__trial_0__geod_method_o1.pt
RsqoResult(success=True, p=tensor([-0.2649, -0.0415,  0.2369]), iters=2, history=RsqoHistory(p_hist=tensor([[-0.3385, -0.0770,  0.1957],
        [-0.2649, -0.0415,  0.2369]]), f_hist=tensor([0.0002, 0.0042]), gs_hist=tensor([[0.0416],
        [0.0084]]), hs_hist=tensor([], size=(2, 0)), g_mults_hist=tensor([[0.],
        [0.]]), h_mults_hist=tensor([], size=(2, 0)), penalty_hist=tensor([1.2000, 1.2000])))
	Saving to ../data/constrained/rsqo/metric_euclid__scaling_1.0__trial_1__geod_metho


Trials: 100%|██████████| 10/10 [00:00<00:00, 69.20it/s][A
Configurations: 2it [00:00, 10.68it/s]

RsqoResult(success=True, p=tensor([-0.2643, -0.0414,  0.2369]), iters=2, history=RsqoHistory(p_hist=tensor([[-0.3398, -0.0752,  0.1996],
        [-0.2643, -0.0414,  0.2369]]), f_hist=tensor([8.8584e-05, 4.2724e-03]), gs_hist=tensor([[0.0412],
        [0.0082]]), hs_hist=tensor([], size=(2, 0)), g_mults_hist=tensor([[0.],
        [0.]]), h_mults_hist=tensor([], size=(2, 0)), penalty_hist=tensor([1.2000, 1.2000])))
	Saving to ../data/constrained/rsqo/metric_euclid__scaling_1.0__trial_0__geod_method_o2.pt
RsqoResult(success=True, p=tensor([-0.2649, -0.0415,  0.2369]), iters=2, history=RsqoHistory(p_hist=tensor([[-0.3385, -0.0770,  0.1957],
        [-0.2649, -0.0415,  0.2369]]), f_hist=tensor([0.0002, 0.0042]), gs_hist=tensor([[0.0416],
        [0.0084]]), hs_hist=tensor([], size=(2, 0)), g_mults_hist=tensor([[0.],
        [0.]]), h_mults_hist=tensor([], size=(2, 0)), penalty_hist=tensor([1.2000, 1.2000])))
	Saving to ../data/constrained/rsqo/metric_euclid__scaling_1.0__trial_1__geod_metho


Trials:   0%|          | 0/10 [00:00<?, ?it/s]

...

RsqoResult(success=True, p=tensor([ 0.1331, -0.5330, -1.7336]), iters=1, history=RsqoHistory(p_hist=tensor([[ 0.1331, -0.5330, -1.7336]]), f_hist=tensor([29.6430]), gs_hist=tensor([[62.3450]]), hs_hist=tensor([], size=(1, 0)), g_mults_hist=tensor([[2.8974e+37]]), h_mults_hist=tensor([], size=(1, 0)), penalty_hist=tensor([2.8974e+37])))
	Saving to ../data/constrained/rsqo/metric_asymmetric__scaling_3.0__trial_7__geod_method_o3_o2.pt
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling b


Trials:  90%|█████████ | 9/10 [00:33<00:03,  3.71s/it]

RsqoResult(success=True, p=tensor([ 0.2385, -1.1901, -1.4470]), iters=1, history=RsqoHistory(p_hist=tensor([[ 0.2385, -1.1901, -1.4470]]), f_hist=tensor([22.0846]), gs_hist=tensor([[47.3935]]), hs_hist=tensor([], size=(1, 0)), g_mults_hist=tensor([[2.9069e+35]]), h_mults_hist=tensor([], size=(1, 0)), penalty_hist=tensor([2.9069e+35])))
	Saving to ../data/constrained/rsqo/metric_asymmetric__scaling_3.0__trial_8__geod_method_o3_o2.pt
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling back to order 1 estimate
failed to find solution to order 2 approx log map, falling b


Trials: 100%|██████████| 10/10 [00:36<00:00,  3.69s/it]
Configurations: 72it [12:14, 10.21s/it]

RsqoResult(success=True, p=tensor([ 0.5698, -0.8710, -1.4893]), iters=1, history=RsqoHistory(p_hist=tensor([[ 0.5698, -0.8710, -1.4893]]), f_hist=tensor([16.8191]), gs_hist=tensor([[34.8212]]), hs_hist=tensor([], size=(1, 0)), g_mults_hist=tensor([[4.9156e+30]]), h_mults_hist=tensor([], size=(1, 0)), penalty_hist=tensor([4.9156e+30])))
	Saving to ../data/constrained/rsqo/metric_asymmetric__scaling_3.0__trial_9__geod_method_o3_o2.pt
